# 🤖 AI Resume Screening System
## Run on Google Colab — Full Working Demo

---

### ✅ Steps to Run:
1. **Run all cells in order** (Runtime → Run all)
2. Wait for Setup to complete (~3-5 min first time)
3. **Click the public ngrok URL** that appears at the end
4. Use the app in browser — upload resumes & job descriptions!

> 💡 **Tip:** Enable GPU for faster model loading → Runtime → Change runtime type → T4 GPU

---

In [ ]:
# ============================================================
# CELL 1: Clone the GitHub Repository
# ============================================================
import os

REPO_URL = "https://github.com/rohangargjfl/AI-Resume_Screening_System.git"
REPO_DIR = "/content/AI_Resume_Screening_System"

if os.path.exists(REPO_DIR):
    print("📁 Repository already exists. Pulling latest changes...")
    !git -C {REPO_DIR} pull
else:
    print("📥 Cloning repository...")
    !git clone {REPO_URL} {REPO_DIR}

print("✅ Repository ready!")
%cd {REPO_DIR}
!ls

In [ ]:
# ============================================================
# CELL 2: Install System Dependencies (Tesseract OCR + Poppler)
# ============================================================
print("📦 Installing system dependencies...")
!apt-get update -q
!apt-get install -y -q tesseract-ocr poppler-utils tesseract-ocr-eng
print("✅ System dependencies installed!")

In [ ]:
# ============================================================
# CELL 3: Install Python Dependencies
# ============================================================
print("🐍 Installing Python packages (this may take 3-5 minutes)...")

!pip install -q flask>=3.0 \
    PyPDF2>=3.0 \
    python-docx>=1.0 \
    spacy>=3.7 \
    scikit-learn>=1.3 \
    matplotlib>=3.8 \
    numpy>=1.24 \
    flask-login>=0.6 \
    flask-sqlalchemy \
    authlib>=1.3 \
    requests>=2.31 \
    pytesseract>=0.3.10 \
    pdf2image>=1.16 \
    pillow>=10.0 \
    torch>=2.0 \
    transformers>=4.40 \
    accelerate>=0.26 \
    huggingface-hub>=0.20.0 \
    python-dotenv \
    werkzeug \
    pyngrok \
    qwen-vl-utils>=0.0.8

print("📚 Downloading spaCy English model...")
!python -m spacy download en_core_web_sm -q

print("✅ All Python packages installed!")

In [ ]:
# ============================================================
# CELL 4: Configure Environment Variables
# ============================================================
import os

# Set working directory
os.chdir("/content/AI_Resume_Screening_System")

# ⚙️ Environment Configuration
# USE_MOCR=false → uses Tesseract OCR (faster, works without GPU too)
# Change to true if you want the advanced VLM-based OCR (needs more GPU RAM)
os.environ['USE_MOCR'] = 'false'          # Set 'true' for advanced OCR (needs GPU)
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['SECRET_KEY'] = 'colab-demo-secret-key-2024'

# Optional: Google OAuth (leave empty for demo login)
os.environ['GOOGLE_CLIENT_ID'] = ''
os.environ['GOOGLE_CLIENT_SECRET'] = ''

print("✅ Environment configured!")
print(f"   USE_MOCR = {os.environ['USE_MOCR']}")
print("   (Demo login enabled — no Google OAuth needed)")

In [ ]:
# ============================================================
# CELL 5: Setup ngrok Tunnel + Start Flask Server
# ============================================================
# ngrok creates a public HTTPS URL so you can access the app from any browser!

import sys
import os
import threading
import time

os.chdir("/content/AI_Resume_Screening_System")
ROOT_DIR = "/content/AI_Resume_Screening_System"
if ROOT_DIR not in sys.path:
    sys.path.insert(0, ROOT_DIR)

# ── ngrok Setup ──────────────────────────────────────────────
from pyngrok import ngrok, conf

# 🔑 NGROK AUTH TOKEN
# Get your FREE token at: https://dashboard.ngrok.com/get-started/your-authtoken
# Paste it below between the quotes:
NGROK_AUTH_TOKEN = ""  # <-- PASTE YOUR TOKEN HERE

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ ngrok authenticated with your token")
else:
    print("⚠️  No ngrok token provided — using anonymous tunnel (may be slower)")
    print("   Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken")

# ── Create Flask App ─────────────────────────────────────────
from dotenv import load_dotenv
load_dotenv(os.path.join(ROOT_DIR, '.env'), override=False)

from flask import Flask
from models import db

app = Flask(
    __name__,
    template_folder=os.path.join(ROOT_DIR, 'web_app', 'templates'),
    static_folder=os.path.join(ROOT_DIR, 'web_app', 'static'),
)
app.secret_key = os.environ.get('SECRET_KEY', 'colab-demo-secret-key-2024')
app.config['MAX_CONTENT_LENGTH'] = 32 * 1024 * 1024  # 32 MB
app.config['SQLALCHEMY_DATABASE_URI'] = 'sqlite:////content/AI_Resume_Screening_System/instance/users_colab.db'
app.config['SQLALCHEMY_TRACK_MODIFICATIONS'] = False
app.config['GOOGLE_CLIENT_ID'] = os.environ.get('GOOGLE_CLIENT_ID', '')
app.config['GOOGLE_CLIENT_SECRET'] = os.environ.get('GOOGLE_CLIENT_SECRET', '')

db.init_app(app)

with app.app_context():
    os.makedirs('/content/AI_Resume_Screening_System/instance', exist_ok=True)
    db.create_all()

from web_app.routes import main
app.register_blueprint(main)

from web_app.auth import init_auth
init_auth(app)

# ── Start Flask in background thread ─────────────────────────
PORT = 5001

def run_flask():
    app.run(host='0.0.0.0', port=PORT, debug=False, use_reloader=False)

flask_thread = threading.Thread(target=run_flask, daemon=True)
flask_thread.start()
time.sleep(2)  # Give Flask time to start

# ── Open ngrok Tunnel ─────────────────────────────────────────
public_url = ngrok.connect(PORT)
print()
print("=" * 60)
print("🚀  AI RESUME SCREENING SYSTEM IS LIVE!")
print("=" * 60)
print(f"🌐  Open this URL in your browser:")
print(f"    {public_url}")
print()
print("📝  Demo Login Instructions:")
print("    1. Click the URL above")
print("    2. Click 'Register' to create a demo account")
print("    3. Enter any name, email & password")
print("    4. Login and start screening resumes!")
print("=" * 60)
print()
print("⚡  To keep running, do NOT close this cell or the Colab tab")
print("💡  The URL changes each time you restart — share the new URL each time")
print()


---

## 📖 How to Use the App

| Step | Action |
|------|--------|
| 1 | Click the **ngrok URL** printed above |
| 2 | Click **Register** → create account (any email/password) |
| 3 | Login with those credentials |
| 4 | Click **Upload** in the navbar |
| 5 | Paste a **Job Description** or upload a file |
| 6 | Upload one or more **Resume PDFs** |
| 7 | Click **Analyze** and wait for results |
| 8 | See ranked candidates with scores, charts & explanations! |

---

## ⚙️ Advanced Options

### Enable Advanced OCR (VLM-based, needs GPU)
In **Cell 4**, change:
```python
os.environ['USE_MOCR'] = 'true'   # ← change false to true
```
Then run all cells again.

### Add Google OAuth Login
In **Cell 4**, add your Google OAuth credentials:
```python
os.environ['GOOGLE_CLIENT_ID'] = 'your_client_id_here'
os.environ['GOOGLE_CLIENT_SECRET'] = 'your_secret_here'
```

---

## 🔑 Getting a Free ngrok Token (Recommended)
1. Go to [https://dashboard.ngrok.com/signup](https://dashboard.ngrok.com/signup)
2. Sign up for free
3. Go to **Your Authtoken** page
4. Copy the token and paste it in **Cell 5** where it says `NGROK_AUTH_TOKEN = ""`

Without a token, ngrok still works but may show a warning page before your app.

---

## ❓ Troubleshooting

| Problem | Solution |
|---------|----------|
| `ModuleNotFoundError` | Re-run Cell 3 |
| Tunnel URL not working | Re-run Cell 5 |
| OCR not working | Check Tesseract: `!which tesseract` |
| Out of memory | Change runtime type to T4 GPU |
| Session expired | Run Cell 5 again to restart the server |
